In [2]:
# static model

# shortcut
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:gpt-oss:20b",
    system_prompt="You are an AI assistant",
)

# explicit chat/llm object
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="gpt-oss:20b",
    temperature=0,
    reasoning=False
    # other params...
)
agent = create_agent(
    llm,
    system_prompt="You are an AI assistant",
)

In [3]:
# dynamic model
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


basic_model = ChatOllama(model="granite4:3b", temperature=0, reasoning=False)
advanced_model = ChatOllama(model="nemotron-3-nano:30b", temperature=0, reasoning=True)

@wrap_model_call
def dynamic_model_selection(
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.messages)
    if message_count // 2 >= 2:
        # Use an advanced model for longer conversations
        model = advanced_model
        print(f"Using [{advanced_model.model}] for complex conversation.")
    else:
        model = basic_model
    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    middleware=[dynamic_model_selection]
)

queries = [
    "What is the capital of France?",
    "Tell me a joke.",
    "What are the implications of quantum computing on modern cryptography?"    
]

history = []
for query in queries:
    history.append({"role": "user", "content": query})
    response = agent.invoke({"messages": history})
    print(f"User: {query}\nAgent: {response['messages'][-1].content}\n")
    history.append({"role": "assistant", "content": response['messages'][-1].content})



User: What is the capital of France?
Agent: The capital of France is Paris.

User: Tell me a joke.
Agent: Sure, here's a classic one for you:

Why don't scientists trust atoms?

Because they make up everything!

Using [nemotron-3-nano:30b] for complex conversation.
User: What are the implications of quantum computing on modern cryptography?
Agent: ### Overview  

Quantum computing threatens the mathematical foundations of most public‑key schemes that protect today’s Internet, banking, digital signatures, and many other security‑critical systems.  The key point is **how a sufficiently large, fault‑tolerant quantum computer (FTQC) would change the hardness assumptions** behind cryptographic primitives:

| Cryptographic primitive | Classical hardness assumption | Quantum algorithm that attacks it | Resulting vulnerability |
|--------------------------|-------------------------------|------------------------------------|--------------------------|
| RSA / PKCS#1 v1.5, RSA‑OAEP | Integer fa

In [1]:
import re
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

# Different models for different complexity levels
fast_model = ChatOllama(model="granite4:3b", temperature=0)
balanced_model = ChatOllama(model="gpt-oss:20b", temperature=0)
advanced_model = ChatOllama(model="nemotron-3-nano:30b", temperature=0.3, reasoning=True)

def assess_query_complexity(query: str) -> str:
    """Classify query complexity based on content analysis."""
    query_lower = query.lower()
    
    # Complex indicators
    complex_patterns = [
        r'\b(compare|contrast|analyze|evaluate|assess|critique)\b',
        r'\b(implications|consequences|ramifications)\b',
        r'\b(why|how come|explain why)\b.*\b(because|since|due to)\b',
        r'\b(design|architect|plan|strategy)\b',
    ]
    
    # Simple indicators
    simple_patterns = [
        r'^\b(what|when|where|who)\b.*\?$',
        r'\b(define|list|name)\b',
    ]
    
    # Check for complex patterns
    if any(re.search(pattern, query_lower) for pattern in complex_patterns):
        return "complex"
    
    # Check for simple patterns
    if any(re.search(pattern, query_lower) for pattern in simple_patterns):
        return "simple"
    
    # Consider length and structure
    word_count = len(query.split())
    if word_count > 30 or '?' in query and len(query.split('?')) > 2:
        return "complex"
    elif word_count < 10:
        return "simple"
    
    return "balanced"

@wrap_model_call
def complexity_based_routing(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Route to appropriate model based on query complexity."""
    last_message = request.messages[-1].content if request.messages else ""
    complexity = assess_query_complexity(last_message)
    
    model_map = {
        "simple": (fast_model, "fast"),
        "balanced": (balanced_model, "balanced"),
        "complex": (advanced_model, "advanced")
    }
    
    model, model_name = model_map[complexity]
    print(f"Complexity: {complexity} → Using {model_name} model [{model.model}]")
    
    return handler(request.override(model=model))

agent = create_agent(
    model=balanced_model,  # Default model
    middleware=[complexity_based_routing]
)
queries = [
    "What is the capital of France?",
    "List the primary colors.",
    "Compare the economic policies of the last two decades.",
    "Explain the implications of artificial intelligence on job markets.",
    "Who wrote 'To Kill a Mockingbird'?"
]

history = []
for query in queries:
    history.append({"role": "user", "content": query})
    response = agent.invoke({"messages": history})
    print(f"User: {query}\nAgent: {response['messages'][-1].content}\n")
    history.append({"role": "assistant", "content": response['messages'][-1].content})

Complexity: simple → Using fast model [granite4:3b]
User: What is the capital of France?
Agent: The capital of France is Paris.

Complexity: simple → Using fast model [granite4:3b]
User: List the primary colors.
Agent: The primary colors are red, blue, and yellow. These colors cannot be created by mixing other colors together and they form the basis for all other colors in the color wheel.

Complexity: complex → Using advanced model [nemotron-3-nano:30b]
User: Compare the economic policies of the last two decades.
Agent: Below is a broad‑brush comparison of the major economic‑policy trends that have unfolded over the **last two decades** (roughly 2005‑2015 and 2015‑2025). Because “the last two decades” can refer to many countries, I’ll illustrate the contrast with three representative economies—**the United States, the European Union, and a typical emerging market (e.g., India)**—and then highlight some common global shifts. Feel free to let me know if you’d like a deeper dive into a s

In [7]:
# dynamic model by context
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable, Literal
import json
from IPython.display import display, Image
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    user_id: str = ""
    user_name: str = "anonymous"
    user_email: str = ""
    user_role: Literal['admin', 'contributor', 'viewer', 'guest'] = "guest"  # admin, contributor, viewer, guest
    user_preferences: dict = None
    def __str__(self):
        return json.dumps(self.__dict__, indent=2)

#mock user data, e.g. from SSO
def _create_context() -> RuntimeContext:
    return RuntimeContext(
        user_id="001",
        user_name="Alice Johnson",
        user_email="alice.johnson@example.com",
        user_preferences={"theme": "dark", "language": "en"}
    )

RUNTIME_CONTEXT = _create_context()
SYS_PROMPT = "You are an AI assistant."
basic_model = ChatOllama(model="granite4:3b", temperature=0, reasoning=False,num_gpu=0)
advanced_model = ChatOllama(model="gpt-oss:20b", temperature=0, reasoning=True)

@wrap_model_call
def dynamic_model_selection(
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Choose model based on context."""
    # access context 
    ctx: RuntimeContext = request.runtime.context
    if ctx and ctx.user_role != 'guest':
        model = advanced_model        
    else:
        model = basic_model
    print(f"Using [{model.model}] for user role `{ctx.user_role}`.")        
    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    system_prompt=SYS_PROMPT,
    context_schema=RuntimeContext,
    middleware=[dynamic_model_selection]
)

def _invoke():
    query = "What are the implications of quantum computing on modern cryptography?"        
    response = agent.invoke({"messages": [{"role": "user", "content": query}]}, context=RUNTIME_CONTEXT)
    print(f"User: {query}\nAgent: {response['messages'][-1].content}\n")

# basic model
_invoke()

# advanced model
RUNTIME_CONTEXT.user_role = 'admin'
_invoke()


Using [granite4:3b] for user role `guest`.
User: What are the implications of quantum computing on modern cryptography?
Agent: Quantum computing represents a significant shift in computational capabilities that has profound implications for modern cryptography, which is fundamentally based on mathematical problems that are easy to perform but hard to reverse without specific information (the "key"). Here are several key implications:

### 1. Threat to Current Cryptographic Algorithms

- **Shor's Algorithm**: One of the most significant threats posed by quantum computing is Shor's algorithm, developed by Peter Shor in 1994. This algorithm can factor large numbers and compute discrete logarithms efficiently on a sufficiently powerful quantum computer. Many current cryptographic systems, such as RSA (Rivest-Shamir-Adleman) and ECC (Elliptic Curve Cryptography), rely on the difficulty of these problems for their security. If a large-scale quantum computer were to be built, it could potenti

In [4]:
# tools: wrapper for monitoring tool calls
from langchain.agents import create_agent
from langchain.tools import tool  
from langchain.tools.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langgraph.types import Command
from langchain_community.retrievers import WikipediaRetriever, ArxivRetriever
from typing import Callable

def _retrieve(retriever, query: str):
    """Run retriever in thread pool to avoid blocking"""
    retrieved_docs = retriever.invoke(query)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs     
@tool(response_format="content_and_artifact")
def wikipedia(query: str):
    """Retrieve information from Wikipedia for general knowledge and fact checking."""
    return _retrieve(WikipediaRetriever(
    lang="en", #language of the articles
    top_k_results=2, #max results to return
    load_max_docs=2, #max downloaded documents
    load_all_available_meta=False, #Published,Title,Summary
    ), query)
@tool(response_format="content_and_artifact")
def arxiv(query: str):
    """Retrieve information from arXiv for academic research papers."""
    return _retrieve(ArxivRetriever(
    top_k_results=2, #max results to return
    load_max_docs=2, #max downloaded documents   
    load_all_available_meta=False, #Published,Title,Authors,Summary
    get_full_documents=True #fetch full text of the papers
    ), query)

@wrap_tool_call
def monitor_tool(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    print(f"Executing tool: {request.tool_call['name']}")
    print(f"Arguments: {request.tool_call['args']}")
    try:
        result = handler(request)
        print(f"Tool completed successfully")
        return result
    except Exception as e:
        print(f"Tool failed: {e}")
        raise

agent = create_agent(
    model="ollama:gpt-oss:20b",
    tools=[wikipedia, arxiv],
    middleware=[monitor_tool],
    system_prompt="You are an AI assistant",
)

queries = [
    "Who is Albert Einstein?",
    "Summarize the latest research on quantum computing from arXiv."    
]

for query in queries:
    response = agent.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"User: {query}\nAgent: {response['messages'][-1].content}\n")

Executing tool: wikipedia
Arguments: {'query': 'Albert Einstein'}
Tool completed successfully
User: Who is Albert Einstein?
Agent: **Albert Einstein (14 March 1879 – 18 April 1955)** was a German‑born theoretical physicist best known for developing the theory of relativity, which revolutionized our understanding of space, time, and gravity.  In 1905 he published four seminal papers—covering the photoelectric effect, Brownian motion, special relativity, and the equivalence of mass and energy ( \(E=mc^2\)\)—earning him the 1921 Nobel Prize in Physics.  Einstein’s later work on general relativity, quantum theory, and statistical mechanics profoundly shaped 20th‑century physics.  He spent much of his career in Switzerland, Germany, and the United States, where he became a naturalized American citizen in 1940.  Einstein’s legacy extends beyond science: his name is attached to a chemical element (Einsteinium), various awards, and the popular image of the “genius” scientist.

Executing tool: 

In [ ]:
!uv pip install -qU duckduckgo-search langchain-tavily
# register tavily key: https://app.tavily.com/home > root .env file > TAVILY_API_KEY="your_key_here"

In [5]:
#dynamic tool selection
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool  
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_tavily import TavilySearch
import os
from dotenv import load_dotenv

@tool()
def duckduckgo_search(query: str):
    """Perform a web search using DuckDuckGo."""
    search = DuckDuckGoSearchResults(num_results=3)
    results = search.invoke(query)
    # DuckDuckGo returns a string, just return it directly
    return results

@tool()
def tavily_search(query: str):
    """Perform a web search using Tavily."""
    search = TavilySearch(max_results=3,topic="general")
    results = search.invoke(query)
    # Tavily returns a list of dicts, format them
    if isinstance(results, list):
        formatted_results = "\n".join(
            f"- {result.get('title', 'N/A')}: {result.get('url', result.get('link', 'N/A'))}" 
            for result in results[:3]
        )
        return formatted_results
    return str(results)

def available_tools():
    return [duckduckgo_search, tavily_search]

def _select_toolset(rq: ModelRequest) -> list[tool]:
    tools = available_tools()
    #rq.runtime.context or rq.runtime.state can be used for more complex logic
    if os.environ.get("TAVILY_API_KEY", "") == "":
        # Remove Tavily if API key is not set
        tools = [tool for tool in tools if tool.name != tavily_search.name]
    else:
        # Remove DuckDuckGo if Tavily is available
        tools = [tool for tool in tools if tool.name != duckduckgo_search.name]    
    return tools

@wrap_model_call
def tool_selector(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Select tools based on query content."""
    return handler(request.override(tools=_select_toolset(request))) 

model = ChatOllama(model="granite4:3b", temperature=0, reasoning=False)
agent = create_agent(
    model=model,
    tools=available_tools(),
    system_prompt="You are an AI assistant, use web search to answer user queries.",
    middleware=[tool_selector]
)
   
def _stream():
    query = "What are the latest advancements in renewable energy?"
    for step in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()

# Simulate missing key - DuckDuckGo will be used
os.environ["TAVILY_API_KEY"] = ""   
_stream()


print("\n---\n")

load_dotenv('../.env', override=True)  # override=True forces reload
_stream()


/home/admin/_github/massimodipaolo/ai-crash-course/.venv/lib/python3.12/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/home/admin/_github/massimodipaolo/ai-crash-course/.venv/lib/python3.12/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


================================ Human Message =================================

What are the latest advancements in renewable energy?
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (4f7c6b72-5e3c-4ef9-8a35-cb1a408ca36d)
 Call ID: 4f7c6b72-5e3c-4ef9-8a35-cb1a408ca36d
  Args:
    query: latest advancements in renewable energy
================================= Tool Message =================================
Name: duckduckgo_search

snippet: This article explores the latest research and innovations in renewable energy technologies, highlighting breakthroughs in solar power, wind energy , and hydrogen production. These advancements not only promise to enhance the efficiency and..., title: Advancements in Renewable Energy Technologies, link: https://www.linkedin.com/pulse/advancements-renewable-energy-technologies-latest-research-kumar-egmoc, snippet: This integration allows for more efficient use of renewable energy sources,